In [57]:
import os 
from minsearch import Index
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv() # get api key 

openai_client = OpenAI(
    api_key=os.getenv("CEREBRAS_API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

In [2]:
# a fixed pipeline 
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
# Agent alternative 
# asking without the tools 
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages=messages,
)

response.choices[0].message.content

'I would love to help you with that, but I do not have access to information about your specific account, the platform you are using, or which course you are referring to.\n\nTo see if you can still join, please check the following on the website or platform where you found the course:\n\n1.  **Enrollment Status:** Look for an "Enroll," "Join," or "Register" button. If it is greyed out or says "Closed," you may not be able to join right now.\n2.  **Dates:** Check if the course has a specific start date or if it is "self-paced." Some courses close enrollment after a certain date.\n3.  **Prerequisites:** Ensure you meet any requirements listed.\n\n**If you can tell me the name of the course or the platform (e.g., Coursera, Udemy, a specific university), I might be able to give you more specific guidance or a link to the sign-up page.**'

In [5]:
# defining the tool
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "function": {  
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [7]:
# sending the question with the tool 
response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages=messages,
    tools=[search_tool],
)
response
# old tool uses function call to point at the need for a fucntion call, now its tools instead.

ChatCompletion(id='chatcmpl-11262604-b093-439b-9ec5-79c92c9a6df6', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='7ec842a4d', function=Function(arguments='{"query": "join course enrollment register"}', name='search'), type='function')], reasoning='The user is asking if they can join a course they just discovered. This is a general question about joining a course, so I should search the FAQ for information about joining, enrollment, or registration. Let me search for relevant terms like "join", "enrollment", "register", etc.'))], created=1782231479, model='zai-glm-4.7', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_a33c042689e969567008', usage=CompletionUsage(completion_tokens=73, prompt_tokens=196, total_tokens=269, completion_tokens_details=Completi

In [8]:
response.choices[0].message.tool_calls[0]


ChatCompletionMessageFunctionToolCall(id='7ec842a4d', function=Function(arguments='{"query": "join course enrollment register"}', name='search'), type='function')

In [9]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [10]:
# 1. Append the model's original message (the one that asked for the tool)
messages.append(response.choices[0].message)

# 2. Append the result in the exact format the API expects
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # Use the ID from the model's call
    "content": result_json         # The string-formatted search results
})

In [11]:
# 3. NOW you can send the updated messages list back to the model
final_response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages=messages,
    tools=[search_tool]
)

In [12]:
final_response

ChatCompletion(id='chatcmpl-9d656a52-1e08-4220-9923-de1a88115993', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Yes, you can still join! According to the FAQ, you're welcome to join and start learning anytime. However, if you want to receive a certificate, you'll need to submit your project while they're still accepting submissions.\n\nGood news - you don't even need to wait for any formal registration or confirmation. You can:\n- Start learning immediately\n- Watch the lesson videos\n- Work through the notebooks and code\n- Submit homework answers through the course platform before the deadline\n\nThe course materials (videos and GitHub resources) are all available, so you can jump right in!", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Great! I found exactly the answer to the user\'s question. The first result is a direct match: "I just discovered the course. Can 

In [18]:
usage = final_response.usage
print(usage)


CompletionUsage(completion_tokens=250, prompt_tokens=946, total_tokens=1196, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=None, reasoning_tokens=132, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=128))


In [ ]:
# price 
output_token = usage.completion_tokens
input_token = usage.prompt_tokens

input_price_per_token = 2.25/1000000
output_price_per_token = 2.75/1000000

total_price = input_token*input_price_per_token + output_token*output_price_per_token
print("usage price is: ", total_price)

usage price is:  0.002816


In [45]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()



def agent_loop(instructions, question, model="zai-glm-4.7") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    while True:
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        msg = response.choices[0].message
        messages.append(msg)  # append the assistant message

        if msg.tool_calls:
            for tc in msg.tool_calls:
                print("function_call:", tc.function.name, tc.function.arguments)
                result = search(**json.loads(tc.function.arguments))
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, indent=2)
                })
        else:
            print("ASSISTANT:", msg.content)
            return msg.content

In [42]:
agent_loop(instructions, "How do I run Olama locally?")

function_call: search {"query": "Olama run locally"}
function_call: search {"query": "Olama installation setup"}
function_call: search {"query": "Ollama"}
ASSISTANT: I assume by "Olama" you meant **Ollama** — the tool for running open-source LLMs locally. Here's how to set it up and run it locally.
### 1. Install Ollama
Visit https://ollama.com/download and choose your operating system:

- macOS: Download the `.pkg` and install it.
- Windows: Download the `.msi` and install it.
- Linux: Run this command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```
### 2. Run a model locally
Once installed, open a terminal and run:
```bash
ollama run llama3
```
This will:
- Download the LLaMA 3 model (~4GB).
- Start the model locally.
- Open a chat interface where you can type questions.
### 3. Test the local Ollama server
Run this command to verify the server is running:
```bash
curl http://localhost:11434
```
You should receive a response like:
```json
{"models": [.

'I assume by "Olama" you meant **Ollama** — the tool for running open-source LLMs locally. Here\'s how to set it up and run it locally.\n### 1. Install Ollama\nVisit https://ollama.com/download and choose your operating system:\n\n- macOS: Download the `.pkg` and install it.\n- Windows: Download the `.msi` and install it.\n- Linux: Run this command in the terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n### 2. Run a model locally\nOnce installed, open a terminal and run:\n```bash\nollama run llama3\n```\nThis will:\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat interface where you can type questions.\n### 3. Test the local Ollama server\nRun this command to verify the server is running:\n```bash\ncurl http://localhost:11434\n```\nYou should receive a response like:\n```json\n{"models": [...]}\n```\n### 4. Use the Python client (optional)\nIf you want to interact with Ollama from Python:\n```bash\npip install ollama\n```\nA 

In [47]:
test = agent_loop(instructions, "I just discovered the course. Can I still join it?")

function_call: search {"query": "join course enrollment late"}
function_call: search {"query": "can I join the course"}
function_call: search {"query": "registration deadline"}
ASSISTANT: Yes, you can still join the course! However, please note that if you want to receive a certificate, you need to submit your project while they're still accepting submissions.

You can start learning whenever you want - the videos and materials are all available. You can just begin following the course materials and submitting homework (while the forms are still open) without needing to wait for any registration confirmation.

Is there anything else about the course you'd like to know?


In [49]:
test = agent_loop(instructions, "what's queen gambit?")

function_call: search {"query": "queen gambit"}
ASSISTANT: I searched the course FAQ database for "queen gambit" but didn't find any results. This question appears to be off-topic and not related to the course or its logistics.

As a course teaching assistant, I can only answer questions about the course material, assignments, deadlines, policies, or other course-related topics. If you have questions about the course itself, please let me know and I'll be happy to help!

Is there anything else about the course that you'd like to explore?


In [71]:
def agent_loop(instructions, question, model="zai-glm-4.7") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    search_count = 0  

    for i in range(10):
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                search_count += 1  # count here
                print(f"search #{search_count}:", tc.function.arguments)
                result = search(**json.loads(tc.function.arguments))
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, indent=2)
                })
        else:
            print(f"Total searches: {search_count}")
            return msg.content

    return msg.content